# Hour 3 — Compare Runs & Find the Winner

By now everyone has at least one `ViT-Tiny finetune` run (Hour 2). In this hour you will:
1. **Clone** your run in the UI, tweak a hyperparameter, and **re-enqueue** it to the `gpu-18gb` queue.
2. **Compare** runs side-by-side in the UI.
3. Use the SDK to **programmatically pick the winner** by validation accuracy.

## 1. Clone → tweak → resubmit (in the Web UI — no code)

1. Open **your** project **`PlantVillage Workshop/<your-name>`** → your `ViT-Tiny finetune` task.
2. Right-click → **Clone** (creates an editable *Draft*).
3. In the Draft: **CONFIGURATION → HYPERPARAMETERS → General**, click **EDIT**, change `learning_rate` (e.g. `1e-3`) or `num_train_epochs`, **SAVE**.
4. Right-click the Draft → **Enqueue** → choose **`gpu-18gb`** (ViT-Tiny fits any tier; `gpu-18gb` has 2 slots. Other tiers: `gpu-35gb`, `gpu-71gb`, `gpu-141gb`).

> **HuggingFace note:** Transformers tasks ignore UI hyperparameter overrides by default. In the cloned task, set `_ignore_hparams_ui_overrides_` = `False` under the **Transformers** hyperparameter group before enqueuing so your edit takes effect.

## 2. Compare side-by-side (in the Web UI)

In the task table, tick **2+** runs → **COMPARE**. Explore:
- **Scalars → Graphs**: overlaid loss / eval accuracy curves.
- **Scalars → Values**: last/min/max accuracy per run, side-by-side.
- **Hyperparameters → Parallel Coordinates**: see which learning_rate/epochs gave the best accuracy.

## 3. Pick the winner programmatically

The same shared credentials + your name as Hour 1 — run this if it's a fresh kernel.

In [1]:
import os
os.environ['CLEARML_WEB_HOST']       = 'http://202.131.110.56'
os.environ['CLEARML_API_HOST']       = 'http://202.131.110.56:8008'
os.environ['CLEARML_FILES_HOST']     = 'http://202.131.110.56:8081'
os.environ['CLEARML_API_ACCESS_KEY'] = ''   # <-- paste the SHARED access key here
os.environ['CLEARML_API_SECRET_KEY'] = ''   # <-- paste the SHARED secret key here
os.environ['WORKSHOP_USER']          = ''   # <-- YOUR name (same as Hour 1)

In [2]:
import os
from clearml import Task

for _k in ('CLEARML_API_ACCESS_KEY', 'CLEARML_API_SECRET_KEY', 'WORKSHOP_USER',
           'CLEARML_WEB_HOST', 'CLEARML_API_HOST', 'CLEARML_FILES_HOST'):
    _v = os.environ.get(_k)
    if _v is not None:
        os.environ[_k] = _v.strip().strip('"').strip("'").strip()
if not (os.environ.get('CLEARML_API_ACCESS_KEY') and os.environ.get('WORKSHOP_USER')):
    try:
        from dotenv import load_dotenv, find_dotenv; load_dotenv(find_dotenv(usecwd=True), override=True)
    except Exception:
        pass
assert os.environ.get('CLEARML_API_ACCESS_KEY') and os.environ.get('WORKSHOP_USER'), \
    'Fill the shared API key AND WORKSHOP_USER (your name) in the credentials cell (or a .env), then re-run.'
PROJECT = f"PlantVillage Workshop/{os.environ['WORKSHOP_USER']}"
# All YOUR completed ViT-Tiny runs.
tasks = Task.get_tasks(
    project_name=PROJECT,
    task_name='ViT-Tiny finetune',
    task_filter={'status': ['completed']},
)
print(f'found {len(tasks)} completed training runs in {PROJECT}')

found 5 completed training runs in PlantVillage Workshop/kartik


In [3]:
# Pull each run's best eval accuracy and rank them.
def best_accuracy(t):
    scalars = t.get_last_scalar_metrics()  # {title: {series: {min/max/last}}}
    # HF Trainer logs eval accuracy under title 'eval' (or 'eval/accuracy'); be tolerant.
    for title, series in scalars.items():
        for name, stats in series.items():
            if 'accuracy' in f'{title}/{name}'.lower():
                return stats.get('max', stats.get('last'))
    return None

ranked = sorted(
    ((t, best_accuracy(t)) for t in tasks),
    key=lambda kv: (kv[1] is not None, kv[1] or 0),
    reverse=True,
)
for t, acc in ranked:
    print(f'{acc!s:>8}  {t.name}  ({t.id})')

   0.995  ViT-Tiny finetune  (e83f2280caf94ac9a15c03c2e7b9f25d)
   0.995  6 epoch: 'ViT-Tiny finetune  (da36ee3e4eb440df8e992612feb72e8b)
    0.99  ViT-Tiny finetune  (056537ed6da14a5d806b22a23d9d6959)
    0.99  ViT-Tiny finetune  (d468dd9c730044e7be67210c3f20fd10)
    0.98  ViT-Tiny finetune  (65a632e218b04f8a9bf0304c40301c32)


In [ ]:
winner, winner_acc = ranked[0]
print(f'winner: {winner.name} ({winner.id}) — accuracy {winner_acc}')
print('output model id:', winner.output_models_id)
print('Publish this model in Hour 4:')
print(f'python workshop/hour4_publish_and_infer.py publish --task-id {winner.id}')

winner: ViT-Tiny finetune (e83f2280caf94ac9a15c03c2e7b9f25d) — accuracy 0.995
output model id: {'model_package': '02d4753172264457a42deed865fedbaa'}
Publish this model in Hour 4:
  python workshop/hour4_publish_and_infer.py publish --task-id e83f2280caf94ac9a15c03c2e7b9f25d


## Lineage

Click the winner's model in the UI → **LINEAGE** tab: it shows the exact task, code, uncommitted changes, packages, dataset, and metrics that produced it. That full provenance is what makes the winner reproducible. On to Hour 4 — publish it and reuse it.